# FCS-BWL-10:  A simple classification task

---



## Import required modules and load data file

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"🐼 pandas     ⇒ {pd.__version__}")
print(f"🤓 numpy      ⇒ {np.__version__}")
print(f"📺 matplotlib ⇒ {plt.matplotlib.__version__}")

In [ ]:
fruits = pd.read_csv("https://drive.switch.ch/index.php/s/wWTGBFrSSCTCphU/download")

In [ ]:
fruits.head(10)

In [ ]:
# create a mapping from fruit label value to fruit name to make results easier to interpret
# here the zip function takes two iterators and creates one new iterator that pairs the n-th elements of the original two
unique_labels = fruits["fruit_label"].unique()
unique_names = fruits["fruit_name"].unique()

label_to_name = dict(zip(unique_labels, unique_names))

label_to_name

In [ ]:
label_to_name[2]

### Create train-test split

In [ ]:
# features
X = fruits[["height", "width", "mass", "color_score"]]

# labels
y = fruits["fruit_label"]

In [ ]:
X.head(2)

In [ ]:
y.head(2)

The file contains the mass, height, and width of a selection of oranges, lemons and apples. The heights were measured along the core of the fruit. The widths were the widest width perpendicular to the height.

In [ ]:
# train_test_split will by default do a split of training data (75%) versus test data (25%)
# the `random_state` is the seed for random number generator: same results every run
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.75, random_state=42
)

In [ ]:
X_train.head()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

In [ ]:
y_train.head(5)

In [ ]:
# let's train the classifier ...
knn.fit(X_train, y_train)

In [ ]:
# ... and see how it performs
score = knn.score(X_test, y_test)

print(f"{score:.3%}")

Is this good?

There is the **DummyClassifier** which can be configured to return based on a few different strategies:

- `most_frequent`: Always return the class with the most examples
- `uniform`: Choose the class randomly (based on uniform distribution)
- ... (see https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

score = dummy.score(X_test, y_test)

print(f"{score:.3%}")

In [ ]:
# Get the predictions from the actual model:
y_pred = knn.predict(X_test)

In [ ]:
y_pred

In [ ]:
label_to_name[3]

In [ ]:
label_to_name[y_test[0]]

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_true=y_test,
    y_pred=y_pred,
    display_labels=unique_names,
    normalize="true",
    cmap="Blues",
)
plt.show()

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_true=y_test, y_pred=y_pred, target_names=unique_names))

* **precision:** from all the observations we have predicted as apples, which are actually apples? $ \rightarrow 67\% $
* **recall:** from all the apples, how many have been predicted as apples? $ \rightarrow 50\% $
* **macro avg** = $(0.67+1.00+0.38+0.50)/4 \approx 0.64 $
* **weighted avg** = $ 0.67 \cdot 4/15 + 1.0 \cdot 2/15 + 0.38 \cdot 4/15 + 0.50 \cdot 5/15 \approx 0.58 $

### Classify new, unseen objects

In [ ]:
new_data = pd.DataFrame(
    [
        {
            "height": 4.3,
            "width": 5.5,
            "mass": 20,
            "color_score": 0.86,
        },
        {
            "height": 8.1,
            "width": 7.8,
            "mass": 97,
            "color_score": 0.35,
        },
    ]
)

new_predictions = knn.predict(new_data)

print("\n".join(label_to_name[p] for p in new_predictions))

## Examining the data

In [ ]:
# Setting colormap throughout notebook
from matplotlib.colors import Normalize, ListedColormap
import matplotlib.patches as mpatches

norm = Normalize(vmin=unique_labels.min(), vmax=unique_labels.max())
label_colors = ["#F7330E", "#5E96C3", "#FF9900", "#CAE00D"]
cmap = ListedColormap(label_colors)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(
    X_train.loc[:, "height"],
    X_train.loc[:, "width"],
    c=y_train,
    s=40,
    marker="o",
    edgecolor="black",
    cmap=cmap,
)
ax.set_xlabel("height")
ax.set_ylabel("width")
handles = [
    mpatches.Patch(color=cmap(norm(label)), label=label_to_name[label])
    for label in unique_labels
]
ax.legend(handles=handles, title="Fruits")

plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(
    X_train.loc[:, "width"],
    X_train.loc[:, "color_score"],
    c=y_train,
    s=40,
    marker="o",
    edgecolor="black",
    cmap=cmap,
)
ax.set_xlabel("width")
ax.set_ylabel("color_score")
handles = [
    mpatches.Patch(color=cmap(norm(label)), label=label_to_name[label])
    for label in unique_labels
]
ax.legend(handles=handles, title="Fruits")

plt.show()

In [ ]:
from mpl_toolkits import mplot3d

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter3D(
    X_train.loc[:, "width"],
    X_train.loc[:, "height"],
    X_train.loc[:, "color_score"],
    c=y_train,
    s=80,
    marker="o",
    edgecolor="black",
    cmap=cmap,
)
ax.set_xlabel("width")
ax.set_ylabel("height")
ax.set_zlabel("color_score")

handles = [
    mpatches.Patch(color=cmap(norm(label)), label=label_to_name[label])
    for label in unique_labels
]
ax.legend(handles=handles, title="Fruits", loc="upper left")

fig.subplots_adjust(left=0, right=1.4, bottom=0.1, top=1.4)

plt.show()

In [ ]:
# Pairwise Feature Scatterplot
import seaborn as sns

data = X_train.copy()
data["label"] = [label_to_name[lbl] for lbl in y_train]

sns.pairplot(
    data,
    hue="label",
    palette=label_colors,
)

plt.show()

## How sensitive is k-NN classification accuracy to the train/test split proportion?

In [ ]:
import sklearn

print(f"{sklearn.__version__ = }")

In [ ]:
# for a nice progress bar
from tqdm.auto import tqdm
from time import sleep

In [ ]:
for _ in tqdm(range(60)):
    pass

In [ ]:
train_sizes = np.arange(0.2, 0.8, 0.1)
num_iterations = 50

fig, ax = plt.subplots()

for train_size in tqdm(train_sizes):
    scores = np.zeros((num_iterations,), dtype=float)

    for i in tqdm(range(num_iterations)):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, train_size=train_size, random_state=i
        )
        knn = KNeighborsClassifier(n_neighbors=5)

        knn.fit(X_train.values, y_train.values)
        scores[i] = knn.score(X_test.values, y_test.values)

    ax.plot(train_size * 100, scores.mean(), "bo")

ax.set_xlabel("Training set proportion (%)")
ax.set_ylabel("Accuracy")

plt.show()

# Draw Decision Boundary

We can visualize the decision boundaries, as seen in the lecture slide.
However, humans lack perception of the 4th dimension, so we only use 2 dimensions: **height** and **width**.

In [ ]:
import matplotlib.patches as mpatches


def draw_decision_boundary(
    k: int,
    x_axis: str = "height",
    y_axis: str = "width",
    unknown_data: pd.DataFrame | None = None,
):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X=X_train[[x_axis, y_axis]], y=y_train)

    x_min = X_train[x_axis].min() - 1
    x_max = X_train[x_axis].max() + 1
    y_min = X_train[y_axis].min() - 1
    y_max = X_train[y_axis].max() + 1

    # This generates all combinations for all (x, y) coordinates
    # The 0.01 defines the resolution: This will take a while!
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, 0.02),
        np.arange(y_min, y_max, 0.02),
    )

    Z_data = pd.DataFrame(
        {
            x_axis: xx.reshape(-1),
            y_axis: yy.reshape(-1),
        }
    )

    Z = knn.predict(Z_data)

    # Put the result into a color plot
    Z = Z.reshape(xx.shape)

    # Plot meshes
    fig, ax = plt.subplots()

    ax.contourf(
        xx,
        yy,
        Z,
        alpha=0.5,
        cmap=cmap,
    )

    # Plot also the training points
    ax.scatter(
        X_train.loc[:, x_axis],
        X_train.loc[:, y_axis],
        c=norm(y_train),
        cmap=cmap,
        s=50,
        edgecolor="k",
    )

    # Plot the new data points
    # if there is new data: predict it as well
    if unknown_data is not None:
        predictions = knn.predict(unknown_data)
        ax.scatter(
            unknown_data.loc[:, x_axis],
            unknown_data.loc[:, y_axis],
            c=norm(predictions),
            cmap=cmap,
            s=100,
            edgecolor="k",
            marker="X",
        )

    ax.set_title(f"KNN Decision Boundary ($ k={k} $)")
    ax.set_xlabel(x_axis)
    ax.set_ylabel(y_axis)
    handles = [
        mpatches.Patch(color=cmap(norm(label)), label=label_to_name[label])
        for label in unique_labels
    ]
    ax.legend(handles=handles, title="Fruits")

    plt.show()

In [ ]:
draw_decision_boundary(k=1)

In [ ]:
unknown_data = pd.DataFrame(
    [
        {
            "height": 4,
            "width": 7,
        },
        {"height": 6, "width": 9},
        {
            "height": 9,
            "width": 10,
        },
        {
            "height": 10,
            "width": 6,
        },
    ]
)

In [ ]:
draw_decision_boundary(k=1, unknown_data=unknown_data)

In [ ]:
draw_decision_boundary(k=5, unknown_data=unknown_data)

In [ ]:
draw_decision_boundary(k=10, unknown_data=unknown_data)

In [ ]:
draw_decision_boundary(k=45, unknown_data=unknown_data)

# Classifier with MinMaxScaler

Many machine learning algorithms perform better when numerical input variables are scaled to a standard range. The two most popular techniques for scaling numerical data prior to modeling are normalization and standardization.

In the following we will use normalization. **Normalization** scales each input variable separately to the range 0-1, which is the range for floating-point values where we have the most precision.

## Scale Matters

Machine learning models learn a mapping from input variables to an output variable.

As such, the scale and distribution of the data drawn from the domain may be different for each variable. Normalization is a rescaling of the data from the original range so that all values are within the new range of 0 and 1.

Normalization requires that you know or are able to accurately estimate the minimum and maximum observable values. You may be able to estimate these values from your available data.

## MinMaxScaler

You can normalize your dataset using the scikit-learn object `MinMaxScaler`.

Good practice usage with the MinMaxScaler and other scaling techniques is as follows:

- Fit the scaler using available training data. For normalization, this means the training data will be used to estimate the minimum and maximum observable values. This is done by calling the fit() function.
- Apply the scale to training data. This means you can use the normalized data to train your model. This is done by calling the transform() function.
- Apply the scale to data going forward. This means you can prepare new data in the future on which you want to make predictions.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)


# we must apply the scaling to the test set that we computed for the training set
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
print(f"Accuracy on training set: {knn.score(X_train_scaled, y_train):.2f}")
print(f"Accuracy on test set:     {knn.score(X_test_scaled, y_test):.2f}")

In [ ]:
scaler.data_min_, scaler.data_max_

In [ ]:
example_fruit = pd.DataFrame(
    [[5.5, 2.2, 10, 0.70]], columns=["height", "width", "mass", "color_score"]
)


example_fruit_scaled = scaler.transform(example_fruit)
print(example_fruit)
print(example_fruit_scaled)
print(f"==> {label_to_name[knn.predict(example_fruit_scaled)[0]]}")

In [ ]:
# Cool thing about scikit-learn:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()


clf.fit(X_train_scaled, y_train)
clf.score(X_test_scaled, y_test)

In [ ]:
# Cool thing about scikit-learn:
from sklearn.svm import LinearSVC

clf = LinearSVC(C=10)

clf.fit(X_train_scaled, y_train)
clf.score(X_test_scaled, y_test)